#### Backtest of pairs trading

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import polars as pl
import polars_ols as pls
import statsmodels.api as sm
from datetime import datetime
from tqdm.notebook import tqdm 

import altair as alt
import matplotlib.pyplot as plt

from jaref_bot.backtest.pair_trading import run_simulation, analyze_strategy

In [ ]:
# Проверка даты первой записи:
token = 'AUDIO'
temp_df = pl.read_parquet(f'./data/{token}_agg_trades.parquet')
temp_df[0]

In [ ]:
results = []
df_results = pl.DataFrame()

start_date = datetime(2024, 3, 1)
end_date = datetime(2025, 4, 1)

sym_1 = 'AUDIO'
sym_2 = 'ICX'

period_arr = ('2h', '4h', '12h')
roll_wind_arr = (7, 10, 15, 30)
dev_in_arr = (2.5, 3.0, 3.5, 4.0)
dev_out_arr = (0.01, 0.25, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0)

total_len = len(dev_in_arr) * len(dev_out_arr) * len(period_arr) * len(roll_wind_arr)

with tqdm(total=total_len) as pbar:
    for period in period_arr:
        for roll_wind in roll_wind_arr:
            for dev_in in dev_in_arr:
                for dev_out in dev_out_arr:
                    
                    df, balance_hist = run_simulation(sym_1=sym_1, 
                                                      sym_2=sym_2,
                                                      period=period,
                                                      roll_wind=roll_wind,
                                                      start_date=start_date,
                                                      end_date=end_date,
                                                      dev_in=dev_in, 
                                                      dev_out=dev_out, 
                                                      balance=10_000, 
                                                      max_order_size=1000, 
                                                      fee_perc=0.0008, 
                                                      verbose=False)
                    print(f'{period=}, {roll_wind=}, {dev_in=}, {dev_out=}', end=' ')
                    metrics = analyze_strategy(balance_hist, start_date, end_date)
                    record = {
                        'period': period,
                        'roll_wind': roll_wind,
                        'dev_in': dev_in,
                        'dev_out': dev_out
                    }
                    record.update(metrics)
                    results.append(record)
                    pbar.update(1)

                    

In [ ]:
res_df = pl.DataFrame(results)
res_df = res_df.drop('start_date', 'end_date', 'initial_balance', 'total_return', 'calmar_ratio',
                     'avg_profit', 'avg_loss', 'n_deals'
    ).filter(pl.col('trades_per_year') > 30
    ).with_columns(
    comb_ratio = (pl.col('annual_return') / abs(pl.col('max_drawdown')) / pl.col('std_return')).cast(pl.Int32)
    )

In [ ]:
res_df.sort(by='sharpe_ratio', descending=True).head(10)#.write_excel('./data/temp.xls')

In [ ]:
best_params = []

for condition in ('annual_return', 'max_drawdown', 'sharpe_ratio', 'sortino_ratio', 'comb_ratio'):
    for row in res_df.sort(by=condition, descending=True).head(10).rows():
        # print(row[:4])
        best_params.append(row[:4])

In [ ]:
len(best_params), len(set(best_params))

In [ ]:
from collections import Counter 
Counter(best_params)

In [ ]:
import seaborn as sns

In [ ]:
pdf = res_df.to_pandas()
pivot_per_sh = pdf.pivot_table(values='sharpe_ratio', index='period', columns='roll_wind', aggfunc='mean')
pivot_dev_sh = pdf.pivot_table(values='sharpe_ratio', index='dev_in', columns='dev_out', aggfunc='mean')

cmap = sns.cubehelix_palette(light=0.9, as_cmap=True)

fig, (ax1, ax2) = plt.subplots(ncols=2, nrows=1, figsize=(9, 3))
ax1.set_title(f'Sharpe ratio')
sns.heatmap(pivot_dev_sh, annot=True, fmt=".1f", ax=ax1, linewidths=.5, cmap=cmap)
sns.heatmap(pivot_per_sh, annot=True, fmt=".1f", ax=ax2, linewidths=.5, cmap=cmap)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(15, 4))
plt.scatter(pdf['std_return'], pdf['annual_return'], alpha=0.6)
plt.xlabel('Риск (стандартное отклонение)')
plt.ylabel('Годовая доходность, %')
plt.title(f'Риск vs Доходность для пары {sym_1} - {sym_2}')

for i, row in pdf.sort_values('annual_return', ascending=False).head(8).iterrows():
    plt.annotate(f"{row['period']}\n{row['roll_wind']}\n{row['dev_in']}\n{row['dev_out']}", 
                 (row['std_return'], row['annual_return']))
plt.tight_layout()

In [ ]:
print(f'{sym_1} - {sym_2}')
pd.set_option("display.precision", 2)
print(pdf.groupby('period')[['sharpe_ratio', 'sortino_ratio', 'annual_return',
    'max_drawdown', 'comb_ratio']].mean().sort_values(by='period', ascending=False).rename(
    columns={'sharpe_ratio': 'sharpe', 'sortino_ratio': 'sortino', 'annual_return': 'annual', 'max_drawdown': 'max_drdwn'}
    ))
print()
print(pdf.groupby('roll_wind')[['sharpe_ratio', 'sortino_ratio', 'annual_return',
    'max_drawdown', 'comb_ratio']].mean().sort_values(by='roll_wind', ascending=True).rename(
    columns={'sharpe_ratio': 'sharpe', 'sortino_ratio': 'sortino', 'annual_return': 'annual', 'max_drawdown': 'max_drdwn'}
    ))
print()
print(pdf.groupby('dev_in')[['sharpe_ratio', 'sortino_ratio', 'annual_return',
    'max_drawdown', 'comb_ratio']].mean().sort_values(by='dev_in', ascending=True).rename(
    columns={'sharpe_ratio': 'sharpe', 'sortino_ratio': 'sortino', 'annual_return': 'annual', 'max_drawdown': 'max_drdwn'}
    ))
print()
print(pdf.groupby('dev_out')[['sharpe_ratio', 'sortino_ratio', 'annual_return',
    'max_drawdown', 'comb_ratio']].mean().sort_values(by='dev_out', ascending=True).rename(
    columns={'sharpe_ratio': 'sharpe', 'sortino_ratio': 'sortino', 'annual_return': 'annual', 'max_drawdown': 'max_drdwn'}
    ))

In [ ]:
[x[0] for x in Counter({('4h', 7, 3.0, 0.01): 7,
         ('2h', 10, 3.5, 0.25): 7,
         ('12h', 7, 3.5, 0.01): 6,
         ('12h', 7, 4.0, 0.01): 5,
         ('12h', 7, 4.0, 0.25): 5,
         ('12h', 7, 4.0, 0.5): 5,
         ('2h', 10, 3.5, 0.5): 5,
         ('4h', 7, 3.5, 0.01): 5,
         ('4h', 7, 3.0, 0.25): 5,
         ('4h', 7, 2.5, 0.01): 4,
         ('4h', 7, 2.5, 0.25): 4,
         ('2h', 10, 3.5, 0.01): 4,
         ('4h', 7, 3.5, 0.5): 4,
         ('4h', 7, 3.0, 0.5): 4,
         ('4h', 7, 4.0, 0.01): 4,
         ('12h', 7, 3.5, 0.25): 4,
         ('12h', 7, 3.5, 0.5): 4,
         ('2h', 10, 4.0, 0.25): 4,
         ('2h', 10, 4.0, 1.0): 4,
         ('2h', 10, 4.0, 1.5): 4,
         ('2h', 10, 4.0, 0.5): 3,
         ('2h', 10, 3.0, 0.01): 3,
         ('4h', 7, 3.5, 0.25): 3,
         ('4h', 10, 4.0, 0.25): 3,
         ('12h', 7, 4.0, 1.0): 3,
         ('2h', 10, 4.0, 0.01): 3,
         ('4h', 10, 4.0, 0.01): 3,
         ('2h', 10, 2.5, 0.25): 3,
         ('2h', 7, 3.0, 3.0): 2,
         ('2h', 7, 2.5, 2.5): 2,
         ('12h', 15, 3.5, 1.0): 2,
         ('4h', 7, 4.0, 0.5): 2,
         ('2h', 10, 3.0, 0.5): 2,
         ('2h', 10, 3.0, 0.25): 2,
         ('12h', 7, 2.5, 3.0): 2,
         ('4h', 7, 3.0, 1.0): 2,
         ('4h', 7, 4.0, 2.0): 2,
         ('4h', 15, 4.0, 0.01): 2,
         ('12h', 10, 3.5, 0.01): 2,
         ('4h', 7, 3.0, 1.5): 2,
         ('4h', 15, 4.0, 3.0): 2,
         ('2h', 10, 2.0, 1.0): 2,
         ('2h', 10, 2.0, 0.25): 2,
         ('2h', 15, 2.0, 0.25): 2,
         ('2h', 15, 3.5, 0.25): 2,
         ('2h', 15, 3.0, 0.25): 2,
         ('2h', 30, 4.0, 2.0): 2,
         ('2h', 7, 3.0, 1.5): 1,
         ('2h', 7, 2.5, 3.0): 1,
         ('2h', 7, 2.5, 1.5): 1,
         ('2h', 7, 2.5, 2.0): 1,
         ('2h', 7, 3.0, 0.25): 1,
         ('2h', 7, 3.0, 0.5): 1,
         ('4h', 10, 3.5, 0.01): 1,
         ('4h', 10, 3.5, 0.25): 1,
         ('4h', 10, 4.0, 0.5): 1,
         ('4h', 7, 2.5, 0.5): 1,
         ('2h', 7, 2.5, 0.01): 1,
         ('12h', 30, 3.0, 1.0): 1,
         ('2h', 10, 2.5, 0.5): 1,
         ('2h', 10, 2.5, 0.01): 1,
         ('4h', 7, 3.0, 2.0): 1,
         ('4h', 7, 2.5, 3.0): 1,
         ('4h', 7, 3.0, 2.5): 1,
         ('12h', 7, 3.0, 3.0): 1,
         ('2h', 30, 3.5, 0.01): 1,
         ('4h', 10, 4.0, 2.0): 1,
         ('4h', 15, 3.5, 0.01): 1,
         ('12h', 10, 3.5, 0.25): 1,
         ('2h', 7, 3.5, 0.25): 1,
         ('4h', 7, 2.5, 1.5): 1,
         ('4h', 10, 2.5, 2.0): 1,
         ('4h', 10, 3.0, 2.0): 1,
         ('2h', 7, 3.0, 2.0): 1,
         ('2h', 7, 3.0, 2.5): 1,
         ('2h', 7, 3.5, 0.01): 1,
         ('12h', 7, 3.0, 0.01): 1,
         ('2h', 7, 3.0, 0.01): 1,
         ('12h', 10, 4.0, 2.5): 1,
         ('2h', 10, 2.0, 2.0): 1,
         ('2h', 10, 2.0, 1.5): 1,
         ('2h', 10, 2.5, 2.5): 1,
         ('2h', 10, 2.5, 1.0): 1,
         ('2h', 10, 3.0, 1.5): 1,
         ('2h', 10, 3.5, 1.5): 1,
         ('2h', 15, 4.0, 0.5): 1,
         ('2h', 30, 4.0, 1.0): 1,
         ('2h', 10, 3.5, 1.0): 1,
         ('2h', 30, 4.0, 2.5): 1,
         ('4h', 15, 4.0, 0.5): 1,
         ('4h', 15, 4.0, 0.25): 1}).items()]

In [10]:
period = '4h'
roll_wind = 7
dev_in = 3.0
dev_out = 0.01

start_date = datetime(2024, 3, 1)
end_date = datetime(2025, 2, 1)

sym_1 = 'AUDIO'
sym_2 = 'LRC'

df, balance_hist = run_simulation(sym_1=sym_1, 
                                  sym_2=sym_2,
                                  period=period,
                                  roll_wind=roll_wind,
                                  start_date=start_date,
                                  end_date=end_date,
                                  dev_in=dev_in, 
                                  dev_out=dev_out, 
                                  balance=10_000, 
                                  max_order_size=1000, 
                                  fee_perc=0.00075, 
                                  verbose=True)
metrics = analyze_strategy(balance_hist, start_date, end_date)
record = {
    'period': period,
    'roll_wind': roll_wind,
    'dev_in': dev_in,
    'dev_out': dev_out
}
record.update(metrics)

[OPEN ORDER] 2024-03-02 13:30:11 buy 1522 LRC at 0.3138, sell 1659 AUDIO at 0.3144
[CLOSE ORDER] 2024-03-02 14:44:12 sell 1522 LRC at 0.31, buy 1659 AUDIO at 0.3048. Balance: 10008.660292699999
[OPEN ORDER] 2024-03-02 14:53:38 buy 1650 AUDIO at 0.3095, sell 1514 LRC at 0.3224
[CLOSE ORDER] 2024-03-02 15:10:13 sell 1650 AUDIO at 0.3099, buy 1514 LRC at 0.3154. Balance: 10018.4275633
[OPEN ORDER] 2024-03-02 17:08:53 buy 1529 LRC at 0.3171, sell 1636 AUDIO at 0.3144
[CLOSE ORDER] 2024-03-02 17:34:51 sell 1529 LRC at 0.3187, buy 1636 AUDIO at 0.3101. Balance: 10026.413398149998
[OPEN ORDER] 2024-03-02 19:10:14 buy 1643 AUDIO at 0.3072, sell 1536 LRC at 0.3218
[CLOSE ORDER] 2024-03-02 23:51:55 sell 1643 AUDIO at 0.3128, buy 1536 LRC at 0.3228. Balance: 10032.571623949998
[OPEN ORDER] 2024-03-03 00:02:10 buy 1200 LRC at 0.322, sell 1956 AUDIO at 0.3132
[CLOSE ORDER] 2024-03-03 00:57:54 sell 1200 LRC at 0.3244, buy 1956 AUDIO at 0.3101. Balance: 10040.019082849996
[OPEN ORDER] 2024-03-03 00:5

In [11]:
record

{'period': '4h',
 'roll_wind': 7,
 'dev_in': 3.0,
 'dev_out': 0.01,
 'start_date': datetime.datetime(2024, 3, 1, 0, 0),
 'end_date': datetime.datetime(2025, 1, 31, 12, 35, 20),
 'total_days': 336,
 'n_deals': 369,
 'initial_balance': 10000,
 'final_balance': 11988,
 'total_return': 19.9,
 'annual_return': 21.8,
 'max_drawdown': -1.6,
 'avg_return': 0.05,
 'std_return': 0.12,
 'trades_per_year': 399.9,
 'sharpe_ratio': 8.33,
 'sortino_ratio': 5.31,
 'calmar_ratio': 13.62,
 'winning_trades': 317,
 'losing_trades': 52,
 'avg_profit': 0.08,
 'avg_loss': -0.12,
 'max_profit': 0.51,
 'max_loss': -1.12,
 'win_ratio': 0.86,
 'avg_usdt_per_deal': 5.39,
 'expected_return': 0.05,
 'profit_factor': 4.12}

#### Общая часть

In [ ]:
from pybit.unified_trading import HTTP
from datetime import datetime, timedelta

import pandas as pd
import polars as pl
import polars.selectors as cs
import numpy as np
from scipy.signal import argrelextrema

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

import ta

import warnings
warnings.filterwarnings("ignore")

In [ ]:
def turn_time_series_to_df(df, token, timeframe='1s'):
    temp_df = df.filter(pl.col('symbol') == token) \
        .select(['date', 'bid_price', 'ask_price', 'bid_size', 'ask_size'])
    
    temp_df = temp_df.with_columns([
        pl.col('bid_price').alias('Open'),
        pl.col('ask_price').alias('High'),
        pl.col('bid_price').alias('Low'),
        pl.col('ask_price').alias('Close'),
        (pl.col('bid_size') + pl.col('ask_size')).alias('Volume')
    ])
    
    # Группировка и агрегация
    temp_df = temp_df.group_by_dynamic('date', every=timeframe).agg([
        pl.first('Open'),
        pl.max('High'),
        pl.min('Low'),
        pl.last('Close'),
        pl.sum('Volume'),
        pl.sum('bid_size').alias('bid_size'),
        pl.sum('ask_size').alias('ask_size')
    ])
    
    return temp_df

In [ ]:
def get_std_lines(df, timeframe_arr, std_arr):
    res_df = df.clone()
    # columns_to_create = []

    for tf in timeframe_arr:
        # res_df = res_df.with_columns(pl.col('Close').rolling_mean(window_size=tf).alias(f'sma_{tf}'))
        # res_df = res_df.with_columns(pl.col('Close').rolling_std(window_size=tf).alias(f'std_{tf}'))
        res_df = res_df.with_columns(pl.col('Close').ewm_mean(span=tf).alias(f'ewma_{tf}'))
        res_df = res_df.with_columns(pl.col('Close').ewm_std(span=tf).alias(f'ewstd_{tf}'))
       
        # for sf in std_arr:
        #     res_df = res_df.with_columns((pl.col(f'sma_{tf}') - sf * pl.col(f'std_{tf}')).alias(f'sma_{tf}-{sf}'))
        #     res_df = res_df.with_columns((pl.col(f'sma_{tf}') + sf * pl.col(f'std_{tf}')).alias(f'sma_{tf}+{sf}'))
        #     res_df = res_df.with_columns((pl.col(f'ewma_{tf}') - sf * pl.col(f'ewstd_{tf}')).alias(f'ewma_{tf}-{sf}'))
        #     res_df = res_df.with_columns((pl.col(f'ewma_{tf}') + sf * pl.col(f'ewstd_{tf}')).alias(f'ewma_{tf}+{sf}'))

    return res_df.drop_nulls()

In [ ]:
def find_pips(data: np.array, n_pips: int, dist_measure: int):
    # dist_measure
    # 1 = Euclidean Distance
    # 2 = Perpindicular Distance
    # 3 = Vertical Distance

    pips_x = [0, len(data) - 1]  # Index
    pips_y = [data[0], data[-1]] # Price

    for curr_point in range(2, n_pips):

        md = 0.0 # Max distance
        md_i = -1 # Max distance index
        insert_index = -1

        for k in range(0, curr_point - 1):
            left_adj = k
            right_adj = k + 1

            time_diff = pips_x[right_adj] - pips_x[left_adj]
            price_diff = pips_y[right_adj] - pips_y[left_adj]
            slope = price_diff / time_diff
            intercept = pips_y[left_adj] - pips_x[left_adj] * slope;

            for i in range(pips_x[left_adj] + 1, pips_x[right_adj]):
                
                d = 0.0 # Distance
                if dist_measure == 1: # Euclidean distance
                    d =  ( (pips_x[left_adj] - i) ** 2 + (pips_y[left_adj] - data[i]) ** 2 ) ** 0.5
                    d += ( (pips_x[right_adj] - i) ** 2 + (pips_y[right_adj] - data[i]) ** 2 ) ** 0.5
                elif dist_measure == 2: # Perpindicular distance
                    d = abs( (slope * i + intercept) - data[i] ) / (slope ** 2 + 1) ** 0.5
                else: # Vertical distance    
                    d = abs( (slope * i + intercept) - data[i] )

                if d > md:
                    md = d
                    md_i = i
                    insert_index = right_adj

        pips_x.insert(insert_index, md_i)
        pips_y.insert(insert_index, data[md_i])

    return pips_x, pips_y

#### Исторические данные, скаченные через API ByBit

In [ ]:
def calculate_rsi(df, window):
    return (
        df.with_columns(
            (pl.col('Close') - pl.col('Close').shift(1)).alias("returns")
            ).with_columns(
                pl.when(pl.col("returns") > 0).then(pl.col("returns")).otherwise(0).alias("gains"),
                pl.when(pl.col("returns") < 0).then(-pl.col("returns")).otherwise(0).alias("losses")
            ).with_columns(
                pl.col("gains").ewm_mean(com=window).alias("avg_gain"),
                pl.col("losses").ewm_mean(com=window).alias("avg_loss")
            ).with_columns(
                (pl.col("avg_gain") / pl.col("avg_loss")).alias("rs")
            ).with_columns(
                (100 - (100 / (1 + pl.col("rs")))).alias("RSI")
            ).drop(["gains", "losses", 'avg_gain', 'avg_loss', 'rs'])
        .filter((pl.col("RSI") > 0) & (pl.col("RSI") < 100))
    )

In [ ]:
def calculate_rsi_signal(df, low_bound, high_bound):
    return (
        df.with_columns(
            pl.when((pl.col("RSI") < low_bound)).then(1).otherwise(0).alias('rsi_entry_signal')
        ).with_columns(
            pl.when((pl.col("RSI") > high_bound)).then(1).otherwise(0).alias('rsi_exit_signal')
        ).with_columns(
            (pl.col("RSI") < low_bound).cum_sum().alias("start_cycle")
        ).with_columns(
            (pl.col("RSI") > high_bound).cum_sum().over("start_cycle").alias("end_cycle")
        ).with_columns(
            pl.when((pl.col("start_cycle") > 0) & (pl.col("end_cycle") == 0))
            .then(1).otherwise(0).alias("rsi_keep_signal")
        ).drop(["start_cycle", "end_cycle"])
    )

In [ ]:
# Timeframe: 5 mins 
# 1. trend detection using 2 EMA curves. Если короткая EMA находится выше длинной - у нас восходящий тренд. И наоборот.
# на восходящем тренде торгуем только лонгами, на низходящем - только шортами
# 2. Bollinger bands for entry signals. Если цена пересекает нижнюю границу - открываем лонг. Если верхнюю - шорт.
# 3. Stop-loss зависит от волатильности рынка. SL = sl_coef * ATR, TP = TPSL_ratio * SL
# 

In [ ]:
pips_x, pips_y = find_pips(df['Close'].values, n_pips=60, dist_measure=1)
temp_df = pd.concat([df['Close'].reset_index(), pd.DataFrame(pips_y, index = pips_x)], axis=1)
temp_df.rename(columns={0: 'Pips'}, inplace=True)

In [ ]:
df = df.merge(temp_df, left_on=[df.index, 'Close'], right_on=['Date', 'Close'])
df.index = df['Date']

In [ ]:
n = 60
df['min'] = df.iloc[argrelextrema(df['Close'].values, np.less_equal, order=n)[0]]['Close']
df['max'] = df.iloc[argrelextrema(df['Close'].values, np.greater_equal, order=n)[0]]['Close']

# Plot results

plt.scatter(df.index, df['min'], c='r')
plt.scatter(df.index, df['max'], c='g')
plt.plot(df.index, df['Close'])
# df['Pips'].dropna().plot()
plt.show()

#### SMA crossing strategy

In [ ]:
# Простая стратегия, когда покупаем, если sma_fast > sma_slow, а продаём, когда наоборот.
# Стоит попробовать менять условия таким образом, чтобы не дожидаться, когда линии пересекутся, а вместо этого
# отслеживать, когда линии начинают сближаться.

In [ ]:
def sma_strategy(df, sma_fast, sma_slow):
    df = df.copy()
    df['return'] = np.log(df['Close'].pct_change() + 1)
    df['sma_fast'] = df['Close'].rolling(sma_fast).mean()
    df['sma_slow'] = df['Close'].rolling(sma_slow).mean()
    df['in_pose'] = np.where(df['sma_fast'] > df['sma_slow'], 1, 0)

    # Сдвигаем на 1 ряд, чтобы учесть время, которое требуется, чтобы открыть позицию
    df['in_pose_return'] = df['in_pose'].shift(1) * df['return']
    df.dropna(inplace=True)
    return df

In [ ]:
def perfomance(df):
    return np.exp(df[['return', 'in_pose_return']].sum()) - 1

In [ ]:
import pandas as pd
import numpy as np

doge_df = pd.read_parquet('./data/doge_historical_data.parquet')
# doge_df = doge_df.resample('60min').ffill()
# doge_df.dropna(inplace=True)
print(doge_df.shape)

In [ ]:
doge_df = sma_strategy(doge_df, 60, 70)

In [ ]:
doge_df['Close'].plot(figsize=(14, 3), title='DOGE/USDT');
doge_df['sma_slow'].plot(label='Slow SMA', legend=True);
doge_df['sma_fast'].plot(label='Fast SMA', legend=True);
doge_df['in_pose'].plot(c='green');

In [ ]:
# Везде, где меняется позиция, мы или покупаем, или продаём актив
doge_df['in_pose'].diff().value_counts()

In [ ]:
n_trades = doge_df['in_pose'].diff().value_counts().iloc[1:].sum()
fees = n_trades * 0.001
n_trades, fees

In [ ]:
print(f'return: {(np.exp(doge_df['return'].sum()) - 1):.6f}')
print(f'strat return: {(np.exp(doge_df['in_pose_return'].sum()) - 1 - fees):.6f}')

In [ ]:
def optimize(df, sma_list1, sma_list2):
    profits = []
    fast, slow = [], []
    # fees = []

    for i, e in zip(sma_list1, sma_list2):
        if i >= e:
            continue
        profit = perfomance(sma_strategy(df, i, e))
        profits.append(profit)
        fast.append(i)
        slow.append(e)

        # n_trades = df['in_pose'].diff().value_counts().iloc[1:].sum()
        # fees.append(n_trades * 0.001)

    frame = pd.DataFrame(profits, [fast, slow]).reset_index().rename(columns={'level_0': 'SMA1', 'level_1': 'SMA2'})
    frame['edge'] = frame['in_pose_return'] - frame['return']
    return frame.sort_values('edge', ascending=False)

In [ ]:
sma_list1 = range(10, 191, 5)
sma_list2 = range(20, 201, 5)

In [ ]:
optimize(doge_df, sma_list1, sma_list2)[:5]

In [ ]:
import backtesting
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
backtesting.set_bokeh_output(notebook=False)

import warnings
warnings.filterwarnings("ignore")

class Indicators_Class():
    def __init__(self, *args, **kwargs):
        super(Indicators_Class, self).__init__(*args, **kwargs)

    def price(arr: pd.Series) -> pd.Series:
        return arr

    def SMA(df: pd.DataFrame, tf: int) -> pd.Series:
        return df['Close'].rolling(tf).mean()


class SMAStrategy(Strategy):
    usd_amount = 10.0
    fast_tf = 7
    slow_tf = 25
    
    def init(self):
        super().init()

        self.price = self.I(Indicators_Class.price, self.data['Close'])
        self.fast_ma = self.I(Indicators_Class.SMA, self.data.df, self.fast_tf)
        self.slow_ma = self.I(Indicators_Class.SMA, self.data.df, self.slow_tf)

    def next(self):
        super().next()
        curr_price = self.price[-1]
        
        if crossover(self.fast_ma, self.slow_ma) and not self.position:
            sl_orig = 0.98 * curr_price
            self.buy(sl=sl_orig, size=self.usd_amount)
        elif crossover(self.slow_ma, self.fast_ma) and self.position.is_long:
            self.position.close()

backtest = Backtest(doge_df, SMAStrategy, commission=0.001, cash=100, exclusive_orders=True)
stats = backtest.run()
print(f"Profit: {stats['Equity Final [$]']-100:.2f}$, Buy and hold: {stats['Buy & Hold Return [%]']:.2f}$, n_trades: {stats['# Trades']:.0f}")

In [ ]:
print(stats[['Buy & Hold Return [%]', 'Return [%]', 'Equity Final [$]', '# Trades', 'Max. Drawdown [%]', 
             'Avg. Drawdown [%]', 'Max. Drawdown Duration', 'Avg. Drawdown Duration']])

In [ ]:
stats, heatmap = backtest.optimize(
    fast_tf = range(10, 91, 5),
    slow_tf = range(30, 240, 10),
    
    maximize = 'Equity Final [$]',
    method = 'grid',
    # maximize = optim_func,
    constraint = lambda param: param.slow_tf > param.fast_tf,
    return_heatmap = True
)
print(stats['_strategy'])
print(f"Profit: {stats['Equity Final [$]']-100:.2f}$, Buy and hold: {stats['Buy & Hold Return [%]']:.2f}$, n_trades: {stats['# Trades']:.0f}")

In [ ]:
stats['_trades'][:5]

In [ ]:
backtest.plot(resample=False);

#### Backtesting

In [ ]:
import backtesting
from backtesting import Backtest, Strategy
from backtesting.lib import crossover, plot_heatmaps, resample_apply, barssince
from backtesting.test import SMA
import numpy as np
import ta
# backtesting.set_bokeh_output(notebook=False)

In [ ]:
class Indicators_Class():
    def __init__(self, *args, **kwargs):
        super(Indicators_Class, self).__init__(*args, **kwargs)

    def stoch(df: pd.DataFrame, tf: int, sw: int) -> pd.Series:
        return ta.momentum.stoch(df['High'], df['Low'], df['Close'], window=tf, smooth_window=sw)

    def rol_d(df: pd.DataFrame, window=3) -> pd.Series:
        return df['%K'].rolling(window).mean()

In [ ]:
class MyStrategy(Strategy):
    usd_amount = 10.0
    tpsl_ratio = 2
    
    macd_slow_w = 30
    macd_fast_w = 8
    
    rsi_w = 55
    rsi_low_bound = 38
    rsi_high_bound = 64

    ema_window = 12
    ema_coef = 0
    
    def init(self):
        super().init()

        self.price = self.data['Close']
        self.rsi = self.I(ta.momentum.rsi, pd.Series(self.price), window=self.rsi_w)
        # self.macd = self.I(ta.trend.macd, pd.Series(self.price), window_slow=self.macd_slow_w, window_fast=self.macd_fast_w)
        # self.macd_signal = self.I(ta.trend.macd_signal, pd.Series(self.price), window_slow=self.macd_slow_w, window_fast=self.macd_fast_w)
        # self.ema = self.I(ta.trend.ema_indicator, pd.Series(self.price), window=self.ema_window)
        
    def next(self):
        super().next()
        curr_price = self.price[-1]
        
        # Закодируем trailing-stop. Используем trailing-stop тогда, когда есть выраженный тренд, чтобы получить
        # преимущество от затяжных стриков. Когда рынок в боковике, не используем.
        # for trade in self.trades:
        #     if trade.is_long:
        #         if self.signal_ma_20[-1] > self.signal_ma_30[-1]:
        #             trade.sl = curr_price * 0.995
        #         else:
        #             trade.sl = max(trade.sl or -np.inf, curr_price - sl_atr * self.sl_coef)
        #     else:
        #         trade.sl = min(trade.sl or np.inf, curr_price + sl_atr * self.sl_coef)

        if (not self.position 
            and self.rsi[-1] < self.rsi_low_bound 
            # and self.macd > 0 
            # and curr_price > self.ema[-1] * self.ema_coef
           ):
            coef = 0.98
            sl_orig = coef * min(curr_price, self.data['Low'][-1])
            # tp_orig = (1 - coef) * self.tpsl_ratio * max(curr_price, self.data['High'][-1]) + max(curr_price, self.data['High'][-1])
            self.buy(sl=sl_orig, size=self.usd_amount)
        if self.position and self.rsi[-1] > self.rsi_high_bound:
            self.position.close()

In [ ]:
def optim_func(series):
    if series["# Trades"] < 20:
        return -1
    return series["Equity Final [$]"]

In [ ]:
backtest = Backtest(main_df.to_pandas(), MyStrategy, commission=0.001, cash=100, exclusive_orders=True)
stats = backtest.run()
print(f"Profit: {stats['Equity Final [$]']-100:.2f}$, Buy and hold: {stats['Buy & Hold Return [%]']:.2f}$, n_trades: {stats['# Trades']:.0f}")

In [ ]:
stats, heatmap = backtest.optimize(
    tpsl_ratio = (1.5, 2, 2.5, 3),
    macd_slow_w = (20, 30, 45, 60, 120),
    macd_fast_w = (10, 15, 20, 30, 45),
    
    rsi_w = (5, 10, 20, 30, 60),
    rsi_low_bound = (25, 30, 35),
    rsi_high_bound = (65, 70, 75),
    ema_window = (5, 10, 20, 40, 100),
    ema_coef = (0, 1),
    
    # maximize = 'Equity Final [$]',
    method = 'grid',
    maximize = optim_func,
    constraint = lambda param: param.macd_slow_w > param.macd_fast_w,
    return_heatmap = True
)

In [ ]:
stats['_strategy']

In [ ]:
stats

In [ ]:
# plot_heatmaps(heatmap, plot_width=1000);

In [ ]:
print(stats['_trades'][:10])

In [ ]:
backtest.plot(resample=False);

In [ ]:
entry_point_idx = stats['_trades']['EntryBar']
exit_point_idx = stats['_trades']['ExitBar']

In [ ]:
df.iloc[exit_point_idx]

#### Собираем датасет для машинного обучения

In [ ]:
# import backtesting
from backtesting import Backtest, Strategy
from backtesting.lib import crossover, barssince
from backtesting.test import SMA
import numpy as np
import ta

In [ ]:
class Indicators_Class():
    def __init__(self, *args, **kwargs):
        super(Indicators_Class, self).__init__(*args, **kwargs)

    def price(arr: pd.Series) -> pd.Series:
        return arr
    
    def EMA(df: pd.DataFrame, n: int) -> pd.Series:
        return ta.trend.ema_indicator(df['Close'], window=n)

    def roll_std(df: pd.DataFrame, n: int) -> pd.Series:
        return df['Close'].rolling(n).std()

    def stoch(df: pd.DataFrame, tf: int, sw: int) -> pd.Series:
        df['%K'] = ta.momentum.stoch(df['High'], df['Low'], df['Close'], window=tf, smooth_window=sw)
        return df['%K']

    def rol_d(df: pd.DataFrame, window=3) -> pd.Series:
        return df['%K'].rolling(window).mean()

    def rsi(df, window):
        return ta.momentum.rsi(df['Close'], window=window)

    def macd(df, window_slow, window_fast):
        return ta.trend.macd_diff(df['Close'], window_slow=window_slow, window_fast=window_fast)

In [ ]:
class MLStrategy(Strategy):
    low_mult = 1.25
    high_mult = 1.25
    tf = 10
    stoch_tf = 20
    stoch_sw = 3
    dw = 3
    rsi_w = 10
    slow_w = 26
    fast_w = 12
    
    
    def init(self):
        super().init()
        self.usd_amount = 10
        self.price = self.I(Indicators_Class.price, self.data['Close'])
        self.ma = self.I(Indicators_Class.EMA, self.data.df, self.tf)
        self.std = self.I(Indicators_Class.roll_std, self.data.df, self.tf)
    
        self.k = self.I(Indicators_Class.stoch, self.data.df, self.stoch_tf, self.stoch_sw)
        self.d = self.I(Indicators_Class.rol_d, self.data.df, self.dw)
        self.rsi = self.I(Indicators_Class.rsi, self.data.df, self.rsi_w)
        self.macd = self.I(Indicators_Class.macd, self.data.df, self.slow_w, self.fast_w)

        # self.last_buy_price = 0

    def next(self):
        super().next()
        curr_price = self.price[-1]
        
        if crossover(self.ma - self.low_mult * self.std, self.price) and not self.position:
            sl_orig = 0.98 * curr_price
            self.buy(sl=sl_orig, size=self.usd_amount)
        if crossover(self.ma + self.high_mult * self.std, self.price) and self.position.is_long:
            self.position.close()


In [ ]:
backtest = Backtest(df, MLStrategy, commission=0.001, cash=100, exclusive_orders=True)
stats = backtest.run()
print(f"Profit: {stats['Equity Final [$]']-100:.2f}$, Buy and hold: {stats['Buy & Hold Return [%]']:.2f}$, n_trades: {stats['# Trades']:.0f}")

In [ ]:
test_size = 0.25
test_idx = int(df.shape[0] * test_size)
# features_df = pd.DataFrame()

train_df = df.copy().iloc[:-test_idx]
test_df = df.copy().iloc[-test_idx:]
print(train_df.shape, test_df.shape)

In [ ]:
from tqdm import tqdm_notebook
from random import choice

In [ ]:
for _ in tqdm_notebook(range(1500)):
    tf = choice([5, 15, 30, 60, 120, 240, 480])
    low_mult = choice((0.6, 0.8, 1.0, 1.25, 1.5, 2.0))
    high_mult = choice((0.6, 0.8, 1.0, 1.25, 1.5, 2.0))
    stoch_tf = choice([5, 15, 30, 60, 120, 240, 480])
    stoch_sw = choice((3, 4, 5, 7, 10))
    dw = choice((3, 4, 5, 7, 10))
    rsi_window = choice([3, 5, 7, 10, 15, 20, 25, 30])
    macd_slow_w = choice([20, 30, 45, 60, 120])
    macd_fast_w = choice([8, 10, 12, 14, 16, 18])
    ema_window = choice([5, 15, 30, 60, 120, 240, 480])
    
    MLStrategy.tf = tf
    MLStrategy.low_mult = low_mult
    MLStrategy.high_mult = high_mult
    MLStrategy.stoch_tf = stoch_tf
    MLStrategy.stoch_sw = stoch_sw
    MLStrategy.dw = dw
    MLStrategy.rsi_w = rsi_window
    MLStrategy.macd_slow_w = macd_slow_w
    MLStrategy.macd_fast_w = macd_fast_w

    tdf = train_df.copy()
        
    tdf['tf'] = tf
    tdf['low_mult'] = low_mult
    tdf['high_mult'] = high_mult
    tdf['rsi_window'] = rsi_window
    tdf['stoch_tf'] = stoch_tf
    tdf['stoch_sw'] = stoch_sw
    tdf['macd_window_slow'] = macd_slow_w
    tdf['macd_window_fast'] = macd_fast_w
    tdf['%K'] = ta.momentum.stoch(tdf['High'], tdf['Low'], tdf['Close'], window=stoch_tf, smooth_window=stoch_sw)
    tdf['%D'] = tdf['%K'].rolling(dw).mean()
    tdf['rsi'] = ta.momentum.rsi(tdf['Close'], window=rsi_window)
    tdf['macd'] = ta.trend.macd_diff(tdf['Close'], window_slow=macd_slow_w, window_fast=macd_fast_w)
    tdf['ema'] = ta.trend.ema_indicator(tdf['Close'], window=ema_window)
    tdf['roll_std'] = tdf['Close'].rolling(ema_window).std()
    tdf.dropna(inplace=True)

    backtest = Backtest(tdf, MLStrategy, commission=0.001, cash=100, exclusive_orders=True)
    stats = backtest.run()
    
    entry_points = stats['_trades']['EntryTime']
    exit_points = stats['_trades']['ExitTime']
    profit = stats['_trades']['PnL']
    sum_profit = profit.sum()

    
    temp_df = tdf.loc[entry_points].reset_index(drop=True)
    temp_df['profit'] = profit.reset_index(drop=True)
    
    features_df = pd.concat([features_df, temp_df], ignore_index=True)
    features_df.to_parquet('./data/features_df_large.parquet')
                
    # print(f'{tf=}, {low_mult=}, {high_mult=}', end=': ')
    # print(f"Profit: {stats['Equity Final [$]']-100:.2f}$, Buy and hold: {stats['Buy & Hold Return [%]']:.2f}$, n_trades: {stats['# Trades']:.0f}")

In [ ]:
# ta.add_all_ta_features(df, open="Open", high="High", low="Low", close="Close", volume="Volume", fillna=True)

In [ ]:
# features_df = pd.read_parquet('./data/features_df.parquet')

In [ ]:
features_df.shape

In [ ]:
features_df.head(3)

In [ ]:
features_df['lower_5m'] = features_df['Close'] < features_df['ema_5m'] - features_df['std_5m']
features_df['lower_15m'] = features_df['Close'] < features_df['ema_15m'] - features_df['std_15m']
features_df['lower_30m'] = features_df['Close'] < features_df['ema_30m'] - features_df['std_30m']
features_df['lower_1h'] = features_df['Close'] < features_df['ema_1h'] - features_df['std_1h']
features_df['lower_4h'] = features_df['Close'] < features_df['ema_4h'] - features_df['std_4h']
features_df['lower_1d'] = features_df['Close'] < features_df['ema_1d'] - features_df['std_1d']

features_df['lower_fast_2'] = features_df['lower_5m'] & features_df['lower_15m']
features_df['lower_fast_3'] = features_df['lower_5m'] & features_df['lower_15m'] & features_df['lower_30m']
features_df['lower_mid_2'] = features_df['lower_1h'] & features_df['lower_4h']
features_df['lower_all'] = features_df['lower_5m'] & features_df['lower_15m'] & features_df['lower_30m'] & features_df['lower_1h'] & features_df['lower_4h'] & features_df['lower_1d']

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(features_df.drop(['profit', 'Open', 'High', 'Low'], axis=1), features_df['profit'],
                                                    test_size=0.25, shuffle=False, random_state=42)


In [ ]:
# mae=0.0340, mape=22.7829
lr = LinearRegression()
lr.fit(X_train, y_train)

preds = lr.predict(X_val)
mae = mean_absolute_error(preds, y_val)
mape = mean_absolute_percentage_error(preds, y_val)
print(f'{mae=:.4f}, {mape=:.4f}')

In [ ]:
# cb_mae=0.0144, cb_mape=4.4155
from catboost import CatBoostRegressor

cb = CatBoostRegressor(n_estimators=500, silent=True)
cb.fit(X_train, y_train)
cb_preds = cb.predict(X_val)
cb_mae = mean_absolute_error(cb_preds, y_val)
cb_mape = mean_absolute_percentage_error(cb_preds, y_val)
print(f'{cb_mae=:.4f}, {cb_mape=:.4f}')

In [ ]:
X_val.shape, preds.shape

In [ ]:
df_val = X_val
df_val['profit'] = y_val
df_val['lr_pred_profit'] = preds
df_val['cb_pred_profit'] = cb_preds

In [ ]:
df_val.head(3)

In [ ]:
df_val['profit'].sum()

In [ ]:
df_val[df_val['lr_pred_profit'] > 0]['profit'].sum()

In [ ]:
df_val[df_val['cb_pred_profit'] > 0]['profit'].sum()

In [ ]:
# Теперь проведём проверку для тестового датасета
# 1. Прогоним тестовый датасет с лучшими параметрами, найденными перебором на первом этапе
# 2. С помощью модели машинного обучения предскажем прогнозируемый profit
# 3. Ещё раз прогоним тестовый датасет, но в качестве сигналов для покупки будет использоваться выход ML.
#    Продаём актив по тому же правилу, что и при первом прогоне

In [ ]:
tf = 30
low_mult = 0.8
high_mult = 2.0
stoch_tf = 30
stoch_sw = 3
dw = 3
rsi_window = 20
macd_slow_w = 26
macd_fast_w = 12
ema_window = 30

MLStrategy.tf = tf
MLStrategy.low_mult = low_mult
MLStrategy.high_mult = high_mult
MLStrategy.stoch_tf = stoch_tf
MLStrategy.stoch_sw = stoch_sw
MLStrategy.dw = dw
MLStrategy.rsi_w = rsi_window
MLStrategy.macd_slow_w = macd_slow_w
MLStrategy.macd_fast_w = macd_fast_w

tdf = test_df.copy()
    
tdf['tf'] = tf
tdf['low_mult'] = low_mult
tdf['high_mult'] = high_mult
tdf['rsi_window'] = rsi_window
tdf['stoch_tf'] = stoch_tf
tdf['stoch_sw'] = stoch_sw
tdf['macd_window_slow'] = macd_slow_w
tdf['macd_window_fast'] = macd_fast_w
tdf['%K'] = ta.momentum.stoch(tdf['High'], tdf['Low'], tdf['Close'], window=stoch_tf, smooth_window=stoch_sw)
tdf['%D'] = tdf['%K'].rolling(dw).mean()
tdf['rsi'] = ta.momentum.rsi(tdf['Close'], window=rsi_window)
tdf['macd'] = ta.trend.macd_diff(tdf['Close'], window_slow=macd_slow_w, window_fast=macd_fast_w)
tdf['ema'] = ta.trend.ema_indicator(tdf['Close'], window=ema_window)
tdf['roll_std'] = tdf['Close'].rolling(ema_window).std()
tdf.dropna(inplace=True)

backtest = Backtest(tdf, MLStrategy, commission=0.001, cash=100, exclusive_orders=True)
stats = backtest.run()

In [ ]:
test_df['Close'].plot(figsize=(14, 3));

In [ ]:
print(stats[['Buy & Hold Return [%]', 'Return [%]', 'Equity Final [$]', '# Trades', 'Max. Drawdown [%]', 
             'Avg. Drawdown [%]', 'Max. Drawdown Duration', 'Avg. Drawdown Duration']])

In [ ]:
X_test = tdf.drop(['Open', 'High', 'Low'], axis=1)
X_test['lower_5m'] = X_test['Close'] < X_test['ema_5m'] - X_test['std_5m']
X_test['lower_15m'] = X_test['Close'] < X_test['ema_15m'] - X_test['std_15m']
X_test['lower_30m'] = X_test['Close'] < X_test['ema_30m'] - X_test['std_30m']
X_test['lower_1h'] = X_test['Close'] < X_test['ema_1h'] - X_test['std_1h']
X_test['lower_4h'] = X_test['Close'] < X_test['ema_4h'] - X_test['std_4h']
X_test['lower_1d'] = X_test['Close'] < X_test['ema_1d'] - X_test['std_1d']

X_test['lower_fast_2'] = X_test['lower_5m'] & X_test['lower_15m']
X_test['lower_fast_3'] = X_test['lower_5m'] & X_test['lower_15m'] & X_test['lower_30m']
X_test['lower_mid_2'] = X_test['lower_1h'] & X_test['lower_4h']
X_test['lower_all'] = X_test['lower_5m'] & X_test['lower_15m'] & X_test['lower_30m'] & X_test['lower_1h'] & X_test['lower_4h'] & X_test['lower_1d']

lr_test_preds = lr.predict(X_test)
cb_test_preds = cb.predict(X_test)

In [ ]:
res_df = tdf
res_df['signal'] = cb_test_preds > 0.01

In [ ]:
res_df['signal'].value_counts()

In [ ]:
class TestMLStrategy(Strategy):
    low_mult = 0.8
    high_mult = 2.0
    tf = 30
    
    def init(self):
        super().init()
        self.usd_amount = 10
        self.price = self.I(Indicators_Class.price, self.data['Close'])
        self.ma = self.I(Indicators_Class.EMA, self.data.df, self.tf)
        self.std = self.I(Indicators_Class.roll_std, self.data.df, self.tf)

    def next(self):
        super().next()
        curr_price = self.price[-1]
        signal = self.data['signal'][-1]
        
        if signal and not self.position:
            sl_orig = 0.98 * curr_price
            self.buy(sl=sl_orig, size=self.usd_amount)
        if crossover(self.ma + self.high_mult * self.std, self.price) and self.position.is_long:
            self.position.close()

In [ ]:
backtest = Backtest(res_df, TestMLStrategy, commission=0.001, cash=100, exclusive_orders=True)
stats = backtest.run()

In [ ]:
print(stats[['Buy & Hold Return [%]', 'Return [%]', 'Equity Final [$]', '# Trades', 'Max. Drawdown [%]', 
             'Avg. Drawdown [%]', 'Max. Drawdown Duration', 'Avg. Drawdown Duration']])

In [ ]:
# Buy & Hold Return [%]: -4.709572

# Naive version: Return [%]: -0.342621 ($99.66), #Trades: 19, Max. Drawdown [%]: -1.11609, Avg. Drawdown [%]: -0.120208
#     Max. Drawdown Duration: 3.5 days, Avg. Drawdown Duration: ~7.5 hours

# Linear regression: Return [%]: -0.39092 ($99.61), #Trades: 31, Max. Drawdown [%]: -1.083198, Avg. Drawdown [%]: -0.122445
#     Max. Drawdown Duration: 6 days, Avg. Drawdown Duration: ~10.5 hours

# Random Forest: Return [%]: -0.363515 ($99.64), #Trades: 12, Max. Drawdown [%]: -0.884796, Avg. Drawdown [%]: -0.132772
#     Max. Drawdown Duration: 6 days, Avg. Drawdown Duration: ~14 hours

# Catboost : Return [%]: 0.425554 ($100.43), #Trades: 15, Max. Drawdown [%]: -0.5721, Avg. Drawdown [%]: -0.068555, 
#     Max. Drawdown Duration: 2 days, Avg. Drawdown Duration: 4 hours

In [ ]:
import bottleneck as bn
import polars as pl
import pandas as pd
import ta

In [ ]:
def read_data(filename, token, timeframe):
    schema = {
        'timestamp': pl.String,
        'symbol': pl.String,
        'bid_price': pl.Float32,
        'ask_price': pl.Float32,
        'bid_size': pl.Float32,
        'ask_size': pl.Float32,
        'volume24h_in_usdt': pl.Float32,
        'last_price': pl.Float32,
        'index_price': pl.Float32,
        'mark_price': pl.Float32,
        'funding_rate': pl.Float32,
        'next_funding_time': pl.Int32
    }
    
    if filename.endswith('.csv'):
        df = pl.scan_csv(filename, schema=schema)
    elif filename.endswith('.parquet'):
        df = pl.scan_parquet(filename)
    else:
        raise NotImplementedError ('Неизвестный формат данных! Файл должен быть или .csv, или .parquet')

    return (
        df.with_columns(
            pl.from_epoch(pl.col("timestamp").cast(pl.Int64) // 1000, time_unit="s")
            .dt.convert_time_zone(time_zone="Europe/Moscow")
        ).rename({"timestamp": "date"})
        .filter(pl.col('symbol') == token)
        .group_by_dynamic('date', every=timeframe).agg([
            pl.first('symbol'),
            pl.first('bid_price').alias('bid_open'),
            pl.max('bid_price').alias('bid_high'),
            pl.min('bid_price').alias('bid_low'),
            pl.last('bid_price').alias('bid_close'),
            pl.sum('bid_size').alias('bid_size'),
            pl.first('ask_price').alias('ask_open'),
            pl.max('ask_price').alias('ask_high'),
            pl.min('ask_price').alias('ask_low'),
            pl.last('ask_price').alias('ask_close'),
            pl.sum('ask_size').alias('ask_size'),
            pl.first('volume24h_in_usdt'),
            pl.mean('last_price'),
            pl.mean('index_price'),
            pl.mean('mark_price'),
            pl.mean('funding_rate'),
            pl.last('next_funding_time')
        ]).with_columns(pl.col('date').dt.cast_time_unit("ms"))
    ).collect()

In [ ]:
def get_std_lines(df, timeframe_arr, std_arr):
    res_df = df.clone()
    for tf in timeframe_arr:
        res_df = res_df.with_columns(pl.col('Close').ewm_mean(span=tf).alias(f'ewma_{tf}'))
        res_df = res_df.with_columns(pl.col('Close').ewm_std(span=tf).alias(f'ewstd_{tf}'))

    return res_df.drop_nulls()

In [ ]:
df = read_data(filename='./data/futures_market_data.csv', token='BTCUSDT', timeframe='60s')

In [ ]:
df['bid_close'].plot()

In [ ]:
# Buy and hold
(df['bid_close'][-1] - df['bid_close'][0]) / df['bid_close'][0]

In [ ]:
tokens = ['ADAUSDT', 'AVAXUSDT', 'XLMUSDT', 'DOGEUSDT', 'OPUSDT', 'APTUSDT', 'XRPUSDT', 'BTCUSDT', 'BNBUSDT']

In [ ]:
total_profit = 0
for token in tokens:
    df = read_data(filename='./data/futures_market_data.parquet', token=token, timeframe='60s')
    buy_and_hold_profit = (df['ask_open'][-1] - df['bid_close'][0]) / df['bid_close'][0] - 0.002
    print(f'{token}. Profit: {buy_and_hold_profit}')
    total_profit += buy_and_hold_profit

print('Total profit:', total_profit)